# Lab 20: Time Series Diagnostics and Advanced Decomposition
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 117 min Core

---

**Learning Objectives:**
- Diagnose a seasonal decomposition of retail sales that does not fit the way the series' seasonal pattern behaves, and fix it
- Diagnose a pair of stationarity tests whose answer about GDP the data cannot support, and read ADF and KPSS together in the 2×2 table
- Separate a daily and a weekly cycle with MSTL, and say what a seasonal part that may change from cycle to cycle takes from the noise
- Put a moving block bootstrap band on an STL trend, and explain why the resampling uses blocks
- Find structural breaks in GDP growth with PELT, and write a function that tests each regime for stationarity
- Ship a module that decomposes a series, tests it, and splits it into regimes

**Dataset:** two FRED series, pulled with your FRED key, stopped at June 2026 and read as FRED published them on 9 October 2026, so that every number below stays the same all term: RSXFSN (U.S. retail sales, not seasonally adjusted, monthly from 2000) and GDPC1 (real GDP, quarterly from 1960). Part 3 uses simulated hourly electricity demand.

**Prerequisites:** your FRED key, saved as the Colab secret `FRED_API_KEY` (Lab 2), and the bootstrap (Lab 7).

**Format:** Parts 1 and 2 each show you an analysis that is wrong and ask what happened. Parts 3 to 5 are run-and-read, with questions. Write it yourself and Part 6 are yours. Write your answers in the text cell under each set of questions.

**Deliverable:** your written answers (Parts 1 to 6), the blanks you fill in Parts 1 and 2, the required `regime_tests()` function, and `src/decompose.py`, a module you finish with your `regime_tests()`, save and upload.

**Time:** Core (117 min, including the required Write it yourself). This lab has no optional section.

With setup (about 10 min) and the README and GitHub submission (about 10), plan on about two and a quarter hours, most of it after class. The minutes are for a first reading: you read every comment and get stuck once or twice.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 1:** one blank, the analyst's decomposition of retail sales, fixed.
> - **Part 2:** two blanks, one in each of the two stationarity tests.
> - **Write it yourself (required, after Part 5):** `regime_tests()`, a function that runs Part 2's two tests on every regime that Part 5's breaks mark out and returns a table, about twelve lines in an empty cell. A check cell tells you when it is right. In Part 6 you paste it, unchanged, into the module.
>
> **You write, in text cells:** two answers in Part 1, two in Part 2, one in Part 3, one in Part 4, two in Part 5 and one in Part 6.
>
> **You run and read:** everything else. That is the two FRED pulls, the analysts' code in Parts 1 and 2 and the made-up series beside it, the 2×2 table and `test_stationarity`, the MSTL decomposition, the block bootstrap, PELT at three penalties and its test of one regime, and the rest of Part 6's module, `decompose.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself turns Part 5's test of one regime into a table of all of them.

## Setup

The first line of the setup cell installs `fredapi` 0.5 (FRED's interface), `ruptures` 1.1 (PELT,
for Part 5) and statsmodels 0.14 or 0.15: the expected numbers in this lab were produced on both.
The cell prints the versions. If it stops with an `ImportError`, or shows another statsmodels
version, use **Runtime → Restart session** and run this cell again. The `warnings.filterwarnings`
lines hide two notices: one Part 2 explains, and one in statsmodels 0.15 about a change planned for a later
version. Other warnings still show.

The second cell reads your FRED key from Colab Secrets, as in Lab 2. If Colab asks whether this
notebook may read `FRED_API_KEY`, allow it. It also sets `END`, the last month this lab uses, and
`VINTAGE`, the day whose published data it asks for. FRED adds a month every few weeks and revises
earlier ones; asking for the data up to June 2026 as FRED had them on 9 October 2026 gives you the
numbers in the expected outputs, whenever you run the lab.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries
!pip install "fredapi>=0.5,<0.6" "statsmodels>=0.14,<0.16" "ruptures>=1.1.9,<1.2" -q

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt
import statsmodels
from fredapi import Fred
from statsmodels.tsa.seasonal import MSTL, STL
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

Path("src").mkdir(exist_ok=True)     # Part 6 writes its module into this folder
warnings.filterwarnings("ignore", category=InterpolationWarning)    # KPSS's p-value notice; Part 2 explains it
warnings.filterwarnings("ignore", message="(adfuller|kpss) currently returns")   # statsmodels 0.15: a future change of return type
print(f"statsmodels {statsmodels.__version__} (this lab needs 0.14 or 0.15), ruptures {rpt.__version__}")
print("Setup complete.")

In [ ]:
# GIVEN — run as-is
# Setup, continued: your FRED key, from Colab Secrets, and the last month this lab uses
# Your FRED key comes from Colab's secret store, never from this cell:
#   left sidebar -> key icon (Secrets) -> + Add new secret
#   Name: FRED_API_KEY    Value: your 32-character key    Notebook access: ON
# Free key: https://fredaccount.stlouisfed.org/apikeys
# You set this ONCE; every notebook on the same Google account then finds it.
FRED_API_KEY = ""
# try/except: try the Colab secret; outside Colab, fall back to an environment variable
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get("FRED_API_KEY") or ""
except Exception:
    import os
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    # No secret set. getpass hides what you type, so the key stays out of this
    # notebook and out of its saved output either way.
    import getpass
    FRED_API_KEY = getpass.getpass("FRED API key (hidden as you type): ").strip()

if len(FRED_API_KEY) < 20:
    raise ValueError(
        "No FRED API key found. Get one free at "
        "https://fredaccount.stlouisfed.org/apikeys, then add it as a Colab "
        "secret named FRED_API_KEY with Notebook access switched ON -- missing "
        "that switch is the most common cause of this message."
    )

fred = Fred(api_key=FRED_API_KEY)
END = "2026-06-01"                   # the last month used
VINTAGE = "2026-10-09"               # the data as FRED published them that day, so later revisions change nothing
print(f"FRED ready. Both series stop at {END}, as published on {VINTAGE}.")

---

## Part 1: DIAGNOSE — Broken STL Decomposition (18 min)

An analyst decomposes U.S. retail sales into a trend, a seasonal pattern and a residual, to say how
large the seasonal pattern is and how it has changed since 2000. This Part repeats the analysis.
Run it, then diagnose it.

**STL** (Seasonal-Trend decomposition using LOESS, from class) writes a series as trend + seasonal
+ residual. It estimates the trend and the seasonal part with LOESS, regressions fitted on nearby
points only, so the trend can bend and the seasonal pattern can change slowly from year to year.
`STL(series, period=12, robust=True)` sets it up for monthly data with a yearly cycle;
`robust=True` gives less weight to outliers such as April 2020; `.fit()` runs it. The result keeps
the parts as `.observed`, `.trend`, `.seasonal` and `.resid`.

**Reading the code.** `fred.get_series` returns a Series dated by month, and `observation_end=END`
stops it at June 2026, and `realtime_start` and `realtime_end` ask for the values FRED published on
`VINTAGE`. `.dropna()` drops any month FRED left empty, `pd.DatetimeIndex(...)` makes
sure the dates are real dates, and `.asfreq("MS")` puts the series on a grid of month starts ("MS"),
so that a month missing from FRED would show up as a gap instead of vanishing.

In [ ]:
# GIVEN — run as-is
# Retail sales (RSXFSN, millions of dollars, not seasonally adjusted), January 2000 to END
retail = fred.get_series("RSXFSN", observation_start="2000-01-01", observation_end=END,
                         realtime_start=VINTAGE, realtime_end=VINTAGE)
retail = retail.dropna()
retail.index = pd.DatetimeIndex(retail.index)
retail = retail.asfreq("MS")          # one row per month start
print(f"{retail.index[0].date()} to {retail.index[-1].date()}: {len(retail)} months")

**Expected output:** `2000-01-01 to 2026-06-01: 318 months`.

**Reading the code.** The analyst measures the seasonal pattern by its *swing*: the peak minus the
trough of the seasonal part within one year. `swing` computes that for any set of numbers, and
`seasonal_swing` applies it to each full year. `[:"2025-12"]` keeps the months up to December 2025,
because 2026 stops in June, before its December peak. `.groupby(seasonal.index.year)` groups the
months by calendar year, and `.agg(swing)` calls `swing` once per group, giving one number per
year. Part 3 uses `swing` again.

In [ ]:
# GIVEN — run as-is
# The seasonal swing: the peak minus the trough of the seasonal part, in each full year
def swing(values):
    """Peak minus trough."""
    return values.max() - values.min()


def seasonal_swing(result):
    """The swing of a decomposition's seasonal part in each calendar year from 2000 to 2025."""
    seasonal = result.seasonal[:"2025-12"]          # 2026 stops in June, before its December peak
    return seasonal.groupby(seasonal.index.year).agg(swing)

**Reading the code.** An STL result's `.plot()` draws the observed series and the three parts one
above another and returns the figure; `fig.set_size_inches(10, 7)` makes it larger. `trend_by_year`
is the mean of the trend in each calendar year, by the same `.groupby`. `swings[2000]` looks up the
year 2000 in the result of `seasonal_swing`, and `/ 1000` turns millions of dollars into billions.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# STL on retail sales, and the seasonal swing beside the trend in six years
stl_result = STL(retail, period=12, robust=True).fit()
fig = stl_result.plot()
fig.set_size_inches(10, 7)
plt.show()

swings = seasonal_swing(stl_result)
trend_by_year = stl_result.trend.groupby(stl_result.trend.index.year).mean()
print("year  seasonal swing  mean trend   ($ billion)")
for year in [2000, 2005, 2010, 2015, 2020, 2025]:
    print(f"{year} {swings[year] / 1000:>14.1f} {trend_by_year[year] / 1000:>11.1f}")
print(f"ratio of the 2025 swing to the 2000 swing: {swings[2025] / swings[2000]:.2f}")

**Expected output:** four panels, then
```
year  seasonal swing  mean trend   ($ billion)
2000           67.9       242.6
2005           86.2       301.3
2010           90.7       312.8
2015          106.3       386.3
2020          109.7       478.4
2025          148.1       611.6
ratio of the 2025 swing to the 2000 swing: 2.18
```

**Your task.** Fill in the blank with a decomposition of retail sales that repairs what the analyst
got wrong. Keep STL with the analyst's `period=12` and `robust=True`: what you change is up to you.
The cell then prints the same seasonal swing for 2000 and 2025, the ratio, and the four panels.

In [ ]:
# YOUR TASK — Part 1: fill in the blank (____)
stl_fixed = ____                    # the decomposition of retail sales, repaired
swings_fixed = seasonal_swing(stl_fixed)
print(f"seasonal swing, 2000: {swings_fixed[2000]:.4f}   2025: {swings_fixed[2025]:.4f}")
print(f"ratio of the 2025 swing to the 2000 swing: {swings_fixed[2025] / swings_fixed[2000]:.2f}")
fig = stl_fixed.plot()
fig.set_size_inches(10, 7)
plt.show()

**Expected output:**
```
seasonal swing, 2000: 0.2728   2025: 0.2449
ratio of the 2025 swing to the 2000 swing: 0.90
```
then four panels. Swings of 0.2793 and 0.2365, a ratio of 0.85, mean that `robust=True` was left out. Swings of 0.1185 and 0.1064 with the same ratio, 0.90, are the right repair in other units: Part 6 and the questions use the units numpy gives by default. A ratio of exactly 1.00 means every year has the same seasonal pattern, which STL does not impose: keep STL.

### YOUR DIAGNOSIS

1. In the analyst's table the seasonal swing, in dollars, more than doubles from 2000 to 2025. Set it beside the mean trend of the same years: what stays about the same? The analyst would report that the seasonal pattern has more than doubled since 2000; your decomposition says it has shrunk by about a tenth. Which statement describes retail sales better, and why?
2. Name what the analyst's decomposition assumes that retail sales do not satisfy, and explain why your change repairs it.

*Your answers here:*

---

## Part 2: DIAGNOSE — Flawed ADF Test (20 min)

Before modelling real GDP, an analyst asks which kind of series its log is. *Trend-stationary*: it
moves around a straight line and its deviations die out, so you remove the line. *Unit root*: its
shocks never die out, so you difference it. The analyst runs the two tests from class. **ADF** has
the null "unit root", **KPSS** the opposite null, "stationary"; each rejects when its p-value is
below 0.05, and together they give the 2×2 table:

| | ADF rejects | ADF does not reject |
|---|---|---|
| **KPSS does not reject** | stationary | inconclusive |
| **KPSS rejects** | contradictory | non-stationary |

**Reading the code.** The first cell pulls GDP as Part 1 pulled retail sales, on a grid of quarter
starts (`"QS"`), and takes its log, because GDP grows at a roughly steady percentage. The second
writes the 2×2 table as a function: each `if` returns as soon as its condition holds.

In [ ]:
# GIVEN — run as-is
# Real GDP (GDPC1, billions of chained dollars, seasonally adjusted), quarterly, 1960 to END, and its log
gdp = fred.get_series("GDPC1", observation_start="1960-01-01", observation_end=END,
                      realtime_start=VINTAGE, realtime_end=VINTAGE)
gdp = gdp.dropna()
gdp.index = pd.DatetimeIndex(gdp.index)
gdp = gdp.asfreq("QS")                # one row per quarter start
log_gdp = np.log(gdp)                 # growth at a steady percentage becomes a straight line
print(f"{gdp.index[0].date()} to {gdp.index[-1].date()}: {len(gdp)} quarters")

In [ ]:
# GIVEN — run as-is
# The 2x2 table from class, as a function of the two p-values
def two_by_two(adf_p, kpss_p, alpha=0.05):
    """The 2x2 table's decision for an ADF and a KPSS p-value at level alpha."""
    if adf_p < alpha and kpss_p >= alpha:
        return "stationary"
    if adf_p >= alpha and kpss_p < alpha:
        return "non-stationary"
    if adf_p < alpha and kpss_p < alpha:
        return "contradictory"
    return "inconclusive"

**Expected output:** `1960-01-01 to 2026-04-01: 266 quarters`.

**Reading the code.** `adfuller` returns six results (the statistic and the p-value first) and
`kpss` four (the same two first); a name of just `_` is a result we do not need. Both take
`regression=`: `"n"` puts nothing besides the series' own past in the test's regression, `"c"` a
constant, `"ct"` a constant and a straight-line trend; the default is `"c"`. KPSS reads its p-value
from a table that stops at 0.01 and 0.10, so `0.0100` means "0.01 or less" and `0.1000` "0.10 or
more" (setup hid the warning that says so).

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# ADF and KPSS on log GDP, and the 2x2 table's decision
adf_stat, adf_p, _, _, _, _ = adfuller(log_gdp, regression="n")
kpss_stat, kpss_p, _, _ = kpss(log_gdp, nlags="auto")
print(f"ADF:  statistic {adf_stat:8.4f}   p-value {adf_p:.4f}")
print(f"KPSS: statistic {kpss_stat:8.4f}   p-value {kpss_p:.4f}")
print(f"Decision at alpha = 0.05: {two_by_two(adf_p, kpss_p)}")

**Expected output:**
```
ADF:  statistic  11.1013   p-value 1.0000
KPSS: statistic   2.5070   p-value 0.0100
Decision at alpha = 0.05: non-stationary
```

**A series whose answer you know.** `trend_demo` is stationary around a line by construction: the
line 8.2 + 0.0075 t, close to log GDP's, plus a deviation of which 80% survives from each quarter
to the next. It has GDP's 266 quarter dates. The cell after it runs the analyst's two calls on it;
`[1]` keeps the second result, the p-value.

In [ ]:
# GIVEN — run as-is
# trend_demo: a straight line plus a deviation that dies out, so stationary around the line
gen = np.random.default_rng(42)       # a generator with a fixed seed, as in Lab 18
n = len(gdp)
shocks = gen.normal(0, 0.01, n)
deviation = np.zeros(n)               # n zeros, filled in by the loop
for t in range(1, n):
    deviation[t] = 0.8 * deviation[t - 1] + shocks[t]     # 80% of last quarter's deviation survives
trend_demo = pd.Series(8.2 + 0.0075 * np.arange(n) + deviation, index=gdp.index)

In [ ]:
# GIVEN — run as-is
# The analyst's two calls, on trend_demo
adf_p_demo = adfuller(trend_demo, regression="n")[1]
kpss_p_demo = kpss(trend_demo, nlags="auto")[1]
print(f"trend_demo: ADF p-value {adf_p_demo:.4f}   KPSS p-value {kpss_p_demo:.4f}   "
      f"decision: {two_by_two(adf_p_demo, kpss_p_demo)}")

**Expected output:** `trend_demo: ADF p-value 1.0000   KPSS p-value 0.0100   decision: non-stationary`.

### YOUR DIAGNOSIS

1. The analyst's tests call `trend_demo` non-stationary, though you built it to be stationary around a line. What is wrong with the two calls, and why could they give no other answer for any series that trends upward?

**Your task.** Fill in the two blanks so that each test can tell the two kinds of series apart. The
cell runs both tests on log GDP and on `trend_demo`.

In [ ]:
# YOUR TASK — Part 2: fill in the two blanks (____)
for name, series in [("log GDP", log_gdp), ("trend_demo", trend_demo)]:
    adf_p_fixed = adfuller(series, regression=____)[1]
    kpss_p_fixed = kpss(series, regression=____, nlags="auto")[1]
    print(f"{name:>10}: ADF p-value {adf_p_fixed:.4f}   KPSS p-value {kpss_p_fixed:.4f}   "
          f"decision: {two_by_two(adf_p_fixed, kpss_p_fixed)}")

**Expected output:**
```
   log GDP: ADF p-value 0.6995   KPSS p-value 0.0100   decision: non-stationary
trend_demo: ADF p-value 0.0000   KPSS p-value 0.1000   decision: stationary
```
`trend_demo` must come out `stationary`, because it was built that way. An ADF p-value of 0.9394 for it means the blanks still give both tests a regression that cannot describe how `trend_demo` was built; `contradictory` or `inconclusive` means the two blanks differ.

### YOUR DIAGNOSIS, continued

2. With your blanks log GDP is still non-stationary, as the analyst said. Was the analyst right, then? Say what your tests can tell you about GDP that theirs could not, and which repair, removing a line or differencing, your result supports.

*Your answers here:*

**The two tests as one function.** `test_stationarity` passes one `regression` to both tests and
returns a dictionary with the two statistics, the two p-values and the 2×2 table's `"decision"`;
Part 5, Write it yourself and Part 6 use it. The cell after it runs it on log GDP, with a trend, and
on GDP growth, which has no trend, with a constant. `.pct_change()` is each quarter's change from
the quarter before as a share of it, `* 100` makes it a percentage, and `.dropna()` drops the first
quarter's empty value.

In [ ]:
# GIVEN — run as-is
# ADF and KPSS with one regression, and the 2x2 table's decision, in one dictionary
def test_stationarity(series, regression="c", alpha=0.05):
    """ADF and KPSS on one series with the same regression, and the 2x2 table's decision."""
    adf = adfuller(series, regression=regression)
    kpss_result = kpss(series, regression=regression, nlags="auto")
    return {"adf_stat": adf[0], "adf_p": adf[1], "kpss_stat": kpss_result[0],
            "kpss_p": kpss_result[1], "decision": two_by_two(adf[1], kpss_result[1], alpha)}

In [ ]:
# GIVEN — run as-is
# Log GDP, with a trend in both tests, and GDP growth, with a constant
gdp_growth = gdp.pct_change().dropna() * 100          # quarterly growth, in percent
for name, series, regression in [("log GDP", log_gdp, "ct"), ("GDP growth", gdp_growth, "c")]:
    result = test_stationarity(series, regression)
    print(f"{name:>10} ({regression}): ADF p-value {result['adf_p']:.4f}   "
          f"KPSS p-value {result['kpss_p']:.4f}   decision: {result['decision']}")

**Expected output:**
```
   log GDP (ct): ADF p-value 0.6995   KPSS p-value 0.0100   decision: non-stationary
GDP growth (c): ADF p-value 0.0000   KPSS p-value 0.0643   decision: stationary
```
Both tests read GDP as I(1): ADF cannot reject a unit root in log GDP, and KPSS rejects
stationarity around a line; GDP growth passes both. Part 5 and Write it yourself work with
`gdp_growth`.

---

## Part 3: EXTEND — MSTL for Multiple Seasonal Periods (10 min)

Retail sales have one seasonal cycle, the year. Hourly electricity demand has two: a daily cycle of
24 hours and a weekly cycle of 168 hours (24 × 7). STL takes one period. **MSTL** (Bandara, Hyndman
and Bergmeir, 2021, in the chapter) takes several: it removes the shortest cycle with STL, then the
next from what is left, and goes round a few times, so that each cycle is estimated with the others
taken out.

You simulate the demand, so you know its true parts: a trend that starts at 500 MW and rises
0.01 MW an hour, a daily wave of amplitude 80 MW (so it swings 160 MW from peak to trough), a
weekly wave of amplitude 40 MW (a swing of 80 MW), and noise with a standard deviation of 15 MW.

**Reading the code.** `np.random.seed(42)` fixes the random draws. `np.sin(2 * np.pi * t / 24)`
is a wave that repeats every 24 hours; subtracting `np.pi / 2` inside it moves the low point to
midnight and the high point to noon. `pd.date_range("2024-01-01", periods=n_hours, freq="h")` gives
one time stamp per hour.

In [ ]:
# GIVEN — run as-is
# 26 weeks of simulated hourly demand: trend + daily wave + weekly wave + noise
np.random.seed(42)
n_hours = 24 * 7 * 26
t = np.arange(n_hours)
trend = 500 + 0.01 * t
daily = 80 * np.sin(2 * np.pi * t / 24 - np.pi / 2)          # lowest at midnight, highest at noon
weekly = 40 * np.sin(2 * np.pi * t / 168 - np.pi)             # one wave every 168 hours
noise = np.random.normal(0, 15, n_hours)
demand_series = pd.Series(trend + daily + weekly + noise, name="demand_MW",
                          index=pd.date_range("2024-01-01", periods=n_hours, freq="h"))
print(f"{len(demand_series)} hours, {demand_series.index[0]} to {demand_series.index[-1]}")

**Expected output:** `4368 hours, 2024-01-01 00:00:00 to 2024-06-30 23:00:00`.

**Reading the code.** `MSTL(demand_series, periods=[24, 168]).fit()` runs MSTL. Its result has
`.observed`, `.trend` and `.resid` like STL's, and a `.seasonal` that is a table with one column per
period, named `seasonal_24` and `seasonal_168`. The chart shows the first two weeks of each part:
`two_weeks` is the positions 0 to 335, and `part.iloc[two_weeks]` keeps them. `zip(axes,
panels.items())` pairs each of the five panels with one name and series from the dictionary.

In [ ]:
# GIVEN — run as-is
# MSTL with a daily (24 h) and a weekly (168 h) period; the first two weeks of each part
mstl_result = MSTL(demand_series, periods=[24, 168]).fit()
two_weeks = slice(0, 24 * 14)
panels = {"observed": mstl_result.observed, "trend": mstl_result.trend,
          "daily (24 h)": mstl_result.seasonal["seasonal_24"],
          "weekly (168 h)": mstl_result.seasonal["seasonal_168"], "residual": mstl_result.resid}
fig, axes = plt.subplots(5, 1, figsize=(10, 9), sharex=True)
for ax, (name, part) in zip(axes, panels.items()):
    ax.plot(part.iloc[two_weeks], linewidth=0.7)
    ax.set_ylabel(name)
plt.show()

print(f"residual std: {mstl_result.resid.std():.2f}")
print(f"daily swing:  {swing(mstl_result.seasonal['seasonal_24']):.1f}")
print(f"weekly swing: {swing(mstl_result.seasonal['seasonal_168']):.1f}")

**Expected output:** five panels, then
```
residual std: 12.24
daily swing:  184.5
weekly swing: 117.9
```

### Interpretation Question

1. Compare MSTL's two swings with the true 160 MW and 80 MW, and its residual standard deviation with the true 15 MW. All three miss in the same direction: where did part of the noise go, and why does a seasonal part that may change from one cycle to the next take some of it?

*Your answers here:*

---

## Part 4: EXTEND — Block Bootstrap for Trend Uncertainty (12 min)

STL gives one trend. How much would it move if history had dealt other shocks? The bootstrap from
Lab 7 answers that kind of question by resampling, but it resamples single observations, which
assumes they are independent. The residuals of a quarterly series are not: a bad quarter tends to
follow a bad quarter. The **moving block bootstrap** (Künsch 1989, in the chapter) resamples
*blocks* of consecutive residuals instead, so the dependence inside each block survives:

1. Decompose log GDP with STL into trend + seasonal + residual.
2. Build a new residual series of the same length from blocks of 8 consecutive residuals (two years), each block starting at a random quarter.
3. Add it to the trend and the seasonal part, and run STL again on the sum.
4. Repeat 200 times. At each quarter, the middle 90% of the 200 trends, from the 5th to the 95th percentile, is a band around the trend.

**Reading the code.** `log_gdp` is Part 2's, so the trend is in log points, and `period=4` is a
yearly cycle in quarterly data. GDPC1 is already
seasonally adjusted; STL still estimates a seasonal part, and the bootstrap keeps it as it is.

In [ ]:
# GIVEN — run as-is
# STL on log real GDP (quarterly, so period=4)
stl_gdp = STL(log_gdp, period=4, robust=True).fit()
fig = stl_gdp.plot()
fig.set_size_inches(10, 7)
plt.show()
print(f"trend from {stl_gdp.trend.min():.3f} to {stl_gdp.trend.max():.3f}, residual std {stl_gdp.resid.std():.4f}")

**Expected output:** four panels, then `trend from 8.149 to 10.104, residual std 0.0088`.

**Reading the code.** `block_resample` fills a new array of n residuals, one block at a time.
`np.random.randint(0, n - block_size + 1)` picks a random first position for a block that fits
inside the series, `resid[start:start + k]` is k consecutive residuals, and the last block is cut
short to fit. `while idx < n:` repeats the body until the array is full.

In [ ]:
# GIVEN — run as-is
# One moving block bootstrap draw: the residuals rebuilt from blocks of consecutive quarters
def block_resample(resid, block_size):
    """A residual series of the same length, made of randomly chosen blocks of block_size residuals."""
    n = len(resid)
    out = np.zeros(n)
    idx = 0
    while idx < n:
        start = np.random.randint(0, n - block_size + 1)     # a block that fits inside the series
        end = min(idx + block_size, n)                        # the last block is cut to fit
        out[idx:end] = resid[start:start + end - idx]
        idx = end
    return out

**Reading the code.** `.values` gives the plain numbers of a Series. `boot_trends` holds one row per
bootstrap trend, and `boot_trends[b] = ...` fills row b. `np.percentile(boot_trends, [5, 95],
axis=0)` takes the two percentiles down each column, that is, at each quarter, and the two rows it
returns are unpacked into two names. The 200 STL fits take a few seconds.

In [ ]:
# GIVEN — run as-is
# 200 bootstrap trends of log GDP, and the band that holds the middle 90% of them at each quarter
np.random.seed(42)
block_size, n_bootstrap = 8, 200                     # 8 quarters = 2 years
resid = stl_gdp.resid.values
boot_trends = np.zeros((n_bootstrap, len(log_gdp)))
for b in range(n_bootstrap):
    boot_series = pd.Series(stl_gdp.trend.values + stl_gdp.seasonal.values
                            + block_resample(resid, block_size), index=log_gdp.index)
    boot_trends[b] = STL(boot_series, period=4, robust=True).fit().trend.values
trend_lower, trend_upper = np.percentile(boot_trends, [5, 95], axis=0)

The band is a few thousandths wide around a trend that runs from 8 to 10, too narrow to see on a
chart of the trend, so the next cell plots its width instead. `log_gdp.index.get_loc(...)` finds the
position of a date.

In [ ]:
# GIVEN — run as-is
# The band's width over time, and at three quarters
ci_width = trend_upper - trend_lower
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(log_gdp.index, ci_width)
ax.set_ylabel("width of the 90% band (log points)")
plt.show()

print(f"mean width {ci_width.mean():.4f}, narrowest {ci_width.min():.4f}, widest {ci_width.max():.4f}")
for quarter in ["2008-10-01", "2019-10-01", "2020-04-01"]:
    print(f"width at {quarter}: {ci_width[log_gdp.index.get_loc(pd.Timestamp(quarter))]:.4f}")

**Expected output:** a chart of the width, then
```
mean width 0.0072, narrowest 0.0039, widest 0.0246
width at 2008-10-01: 0.0088
width at 2019-10-01: 0.0062
width at 2020-04-01: 0.0057
```

### Interpretation Question

1. Why blocks? Say what resampling single residuals, independently, would do to the dependence between neighbouring quarters, and whether you would expect the band to come out wider or narrower.

*Your answers here:*

---

## Part 5: EXTEND — Structural Break Detection + Per-Regime Stationarity (12 min)

Part 2 found GDP growth stationary over 1960–2026 as a whole. A stationarity test asks whether a
series keeps returning to one mean. It does not ask whether that mean, or the spread around it,
changed partway through. **PELT** (Pruned Exact Linear Time, from class) looks for such changes: it
splits the series into regimes, charging a *cost* for how badly each regime fits one distribution
and a *penalty* for each break, and finds the split with the lowest total.

**Reading the code.** `rpt.Pelt(model="rbf")` chooses the cost: `"rbf"` notices a change in the
whole distribution, its mean or its spread (`"l2"`, on the lecture slide, notices only the mean).
`.fit(gdp_growth.values)` hands it the numbers as a plain array, and `.predict(pen=2)` returns
`ends`: the position just after each regime's last quarter, the last entry being the length of the
series. So `gdp_growth.index[end - 1]` is a regime's last quarter, and the list comprehension keeps
it for every regime but the final one. `ax.axvline` draws a vertical line at each break.

In [ ]:
# GIVEN — run as-is
# PELT on quarterly GDP growth: where each regime ends, and the last quarter before each break
ends = rpt.Pelt(model="rbf").fit(gdp_growth.values).predict(pen=2)
break_dates = [gdp_growth.index[end - 1] for end in ends if end < len(gdp_growth)]
print(f"ends: {ends}")
print(f"last quarter before each break: {[str(date.date()) for date in break_dates]}")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(gdp_growth, linewidth=0.8)
for date in break_dates:
    ax.axvline(date, color="red", linestyle="--")
ax.set_ylabel("quarterly growth (%)")
plt.show()

**Expected output:**
```
ends: [95, 160, 265]
last quarter before each break: ['1983-10-01', '2000-01-01']
```
then GDP growth with two dashed lines.

**One regime.** `gdp_growth.iloc[start:end]` keeps the positions from `start` up to `end − 1`, so
`gdp_growth.iloc[0:ends[0]]` is the first regime. The next cell runs Part 2's `test_stationarity`
on it, with its default `regression="c"`, and prints the regime's first and last quarters, its
size, its mean and its standard deviation. `.std()` is pandas' standard deviation, which divides by
n − 1.

In [ ]:
# GIVEN — run as-is
# The first regime alone: its dates, size, mean and spread, and Part 2's two tests
first = gdp_growth.iloc[0:ends[0]]
result = test_stationarity(first)
print(f"{first.index[0].date()} to {first.index[-1].date()}: {len(first)} quarters, "
      f"mean {first.mean():.4f}, std {first.std():.4f}")
print(f"ADF p-value {result['adf_p']:.4f}   KPSS p-value {result['kpss_p']:.4f}   "
      f"decision: {result['decision']}")

**Expected output:**
```
1960-04-01 to 1983-10-01: 95 quarters, mean 0.8585, std 1.0759
ADF p-value 0.0001   KPSS p-value 0.1000   decision: stationary
```

**Three penalties.** The next cell runs the same PELT at `pen=1`, `2` and `3`, and keeps each
result in `pen_ends`, so `ends` stays as it is. `pen_ends[:-1]` drops the last entry, the length of
the series.

In [ ]:
# GIVEN — run as-is
# The same PELT at three penalties: how many breaks, and the last quarter before each
for pen in [1, 2, 3]:
    pen_ends = rpt.Pelt(model="rbf").fit(gdp_growth.values).predict(pen=pen)
    print(f"pen={pen}: {len(pen_ends) - 1} break(s), after "
          f"{[str(gdp_growth.index[end - 1].date()) for end in pen_ends[:-1]]}")

**Expected output:**
```
pen=1: 11 break(s), after ['1973-10-01', '1975-01-01', '1978-10-01', '1980-01-01', '1982-07-01', '1983-10-01', '1990-01-01', '1991-04-01', '2000-01-01', '2020-01-01', '2021-04-01']
pen=2: 2 break(s), after ['1983-10-01', '2000-01-01']
pen=3: 1 break(s), after ['1985-01-01']
```

### Interpretation Questions

1. What happened to the U.S. economy near each of the two `pen=2` break dates? From the chart, is it the level of growth that changes at each break, its volatility, or both?
2. What is the penalty trading off, and why does the 2020 recession appear only at `pen=1`?

*Your answers here:*

---

## Write it yourself (required — 35 min)

This function is required. No AI: write it yourself.

Part 5 tested one regime. In the empty cell below, write a function

`regime_tests(series, ends, min_obs=20)`

that tests every regime and returns a table with one row per regime:

1. `ends` is a list like the one PELT returned: the position just after each regime's last point,
   the last entry being `len(series)`. The first regime starts at position 0, and each later
   regime starts where the one before it ended.
2. Leave out any regime with fewer than `min_obs` points: on a handful of points ADF and KPSS have
   almost no power, and their p-values mean little.
3. For each regime you keep, run Part 2's `test_stationarity` on it, with its default regression,
   and record a dictionary with the keys `"start"` and `"end"` (the regime's first and last
   dates), `"n"` (its number of points), `"mean"` and `"std"` (pandas' `.mean()` and `.std()`),
   and `"adf_p"`, `"kpss_p"` and `"decision"` (from `test_stationarity`'s dictionary).
4. Return `pd.DataFrame(rows)`, where `rows` is the list of your dictionaries: one table row per
   dictionary, as in Lab 18.

Start from Part 5's cell for one regime. A name such as `start`, set to 0 before the loop and to
each regime's end at the bottom of it, carries one regime's end forward as the next one's start.
You need about twelve lines.

The check runs your function on GDP growth with Part 5's `ends`, then on a made-up monthly series of
142 points whose four regimes have 50, 20, 12 and 60 points, with `min_obs` at 20 and at 30. Until
you write the function, the check says "Not written yet"; once you do, it fails with a message
until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: write regime_tests(series, ends, min_obs=20)

In [ ]:
# CHECK — for regime_tests above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "regime_tests" not in globals():
    print("Not written yet: write regime_tests in the empty cell above, then run this check again.")
else:
    import inspect

    def names_used(code):
        """Every global or attribute name a function's code refers to, including inner functions."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def near(a, b):
        """Equal to about six significant digits."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=1e-12))

    def all_near(got, want):
        """Two lists of numbers equal, entry by entry, to about six significant digits."""
        return len(got) == len(want) and all(near(a, b) for a, b in zip(got, want))

    COLUMNS = ["start", "end", "n", "mean", "std", "adf_p", "kpss_p", "decision"]
    KEYS = "adf_stat, adf_p, kpss_stat, kpss_p and decision"

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(regime_tests)
    params = list(signature.parameters)
    assert params[:3] == ["series", "ends", "min_obs"], (
        f"regime_tests's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "regime_tests(series, ends, min_obs=20).")
    assert signature.parameters["min_obs"].default == 20, "Give min_obs a default of 20, as the task says."
    used = names_used(regime_tests.__code__)
    for lab_name in ["gdp_growth", "gdp", "first", "break_dates", "retail", "log_gdp", "trend_demo",
                     "demand_series"]:
        assert lab_name not in used, (
            f"regime_tests uses {lab_name}, a name from the lab, instead of its argument series. "
            "Compute everything from series, ends and min_obs.")
    assert not {"Pelt", "rpt"} & used, (
        "regime_tests runs PELT itself. Use the ends it is given: the check hands it other ends.")
    assert "test_stationarity" in used, (
        "regime_tests does not call test_stationarity. Run Part 2's function on each regime, as "
        "Part 5's cell did, and take adf_p, kpss_p and decision from the dictionary it returns.")

    # Part 2's GDP growth and test_stationarity, and Part 5's ends, as those Parts left them
    try:
        assert len(gdp_growth) == 265 and list(ends) == [95, 160, 265]
        assert near(test_stationarity(gdp_growth)["kpss_p"], 0.0642512541)
    except Exception:
        raise AssertionError(
            "gdp_growth, ends or test_stationarity is no longer what Parts 2 and 5 made: a cell of "
            "yours gave one of those names to something else (rename yours and rerun the notebook from "
            "the top), or your regime_tests changed ends (fix it, then rerun Part 5's PELT cell and "
            "this check).") from None

    def run(where, series, ends_in, min_obs=None):
        """Call regime_tests on copies; return its table with the task's columns, or stop with a message."""
        given = list(ends_in)
        args = (series.copy(), given) + (() if min_obs is None else (min_obs,))
        try:
            got = regime_tests(*args)
        except KeyError as err:
            raise AssertionError(f"{where}, regime_tests stopped with KeyError: {err}. "
                                 f"test_stationarity's dictionary has the keys {KEYS}.") from None
        except TypeError as err:
            if "slice indexing" in str(err) or "DatetimeIndex" in str(err):
                raise AssertionError(
                    f"{where}, regime_tests stopped with TypeError: {err}. .loc takes dates, and ends "
                    "holds positions: slice by position with .iloc, as Part 5's cell did.") from None
            raise AssertionError(f"{where}, regime_tests stopped with TypeError: "
                                 f"{str(err).rstrip('.')}.") from None
        except AttributeError as err:
            if "index" in str(err):
                raise AssertionError(
                    f"{where}, regime_tests stopped with AttributeError: {err}. A regime lost its "
                    "dates: slice the Series itself with .iloc, not its .values.") from None
            raise AssertionError(f"{where}, regime_tests stopped with AttributeError: "
                                 f"{str(err).rstrip('.')}.") from None
        except IndexError as err:
            raise AssertionError(
                f"{where}, regime_tests stopped with IndexError: {err}. It looked up a position past "
                "the end of the series: a regime's last point is at end - 1, and its last date is the "
                "regime's own last date, segment.index[-1].") from None
        except Exception as err:
            raise AssertionError(f"{where}, regime_tests stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        assert given == list(ends_in), (
            f"{where}, regime_tests changes the list of ends it is given (it became {given}). Leave "
            "ends as it is, and keep each regime's start in a name of its own.")
        assert got is not None, f"{where}, regime_tests gives back nothing: use return, not print."
        if isinstance(got, list):
            raise AssertionError(f"{where}, regime_tests gives back a list. Return a table: "
                                 "pd.DataFrame(rows).")
        if not isinstance(got, pd.DataFrame):
            raise AssertionError(f"{where}, regime_tests gives back a {type(got).__name__}, not a table. "
                                 "Return pd.DataFrame(rows).")
        missing = [c for c in COLUMNS if c not in got.columns]
        assert not missing, (
            f"{where}, regime_tests's table has the columns {list(got.columns)}, and lacks "
            f"{missing}. Give each row's dictionary the keys {', '.join(COLUMNS)}, and do not make "
            "any of them the index.")
        return got.reset_index(drop=True)

    def dates(column):
        """A column of dates as Timestamps, or None if it holds something else."""
        try:
            if all(isinstance(x, (int, np.integer)) for x in column):
                return None
            return [pd.Timestamp(x) for x in column]
        except (TypeError, ValueError):
            return None

    # 1. a made-up monthly series with four regimes of 50, 20, 12 and 60 points
    made_up_gen = np.random.default_rng(2026)
    pieces = [made_up_gen.normal(0.5, 1.0, 50), made_up_gen.normal(3.0, 0.4, 20),
              made_up_gen.normal(-1.0, 2.0, 12), np.cumsum(made_up_gen.normal(0.0, 1.0, 60))]
    made_up = pd.Series(np.concatenate(pieces), index=pd.date_range("2010-01-01", periods=142, freq="MS"))
    where = "On the made-up series with min_obs=20"
    table = run(where, made_up, [50, 70, 82, 142])
    sizes = [int(x) for x in table["n"]]
    SIZE_SLIPS = {
        (20, 60): "leave out the first regime, which starts at position 0, before the first entry of "
                  "ends",
        (20, 12, 60): "leave out the first regime, which starts at position 0, and keep the 12-point "
                      "one, which has fewer than min_obs points",
        (50, 20): "stop before the last regime, which ends at len(series), the last entry of ends; or "
                  "the loop stops at the short regime (break) instead of skipping it (continue)",
        (50, 60): "leave out the regime of exactly 20 points. Keep a regime when it has at least "
                  "min_obs points: len(segment) >= min_obs",
        (50, 20, 12, 60): "keep the regime of 12 points. Leave out any regime with fewer than min_obs points",
        (51, 21, 60): "each hold one point too many. A regime runs from its start up to end - 1, which "
                      "is what series.iloc[start:end] keeps",
        (50, 59): "drop the first point of each later regime. The next regime starts at end, not end + 1",
        (49, 59): "drop each regime's first point. A regime starts at the previous end, which "
                  "series.iloc[start:end] keeps",
        (50, 70, 82, 142): "all start at position 0. Start each regime where the one before it ended",
        (50, 20, 72): "run on from a regime you left out: the last one starts where the 12-point "
                      "regime started. Set start = end for regimes you leave out too",
        (142,): "are the whole series, tested once. Cut it at the positions in ends",
    }
    for slip, cause in SIZE_SLIPS.items():
        assert tuple(sizes) != slip, f"{where}, regime_tests gives regimes of {sizes} points. Your regimes {cause}."
    assert sizes == [50, 20, 60], (
        f"{where}, regime_tests gives regimes of {sizes} points; they should be [50, 20, 60] (the "
        "12-point regime is left out). Compare each step with the task: where each regime starts and "
        "ends, and which are left out.")

    starts, last_dates = dates(table["start"]), dates(table["end"])
    assert starts is not None and last_dates is not None, (
        f"{where}, the start and end columns hold {list(table['start'])} and {list(table['end'])}, "
        "not dates. Record each regime's first and last dates: segment.index[0] and segment.index[-1].")
    WANT_STARTS = [pd.Timestamp(d) for d in ["2010-01-01", "2014-03-01", "2016-11-01"]]
    WANT_ENDS = [pd.Timestamp(d) for d in ["2014-02-01", "2015-10-01", "2021-10-01"]]
    assert last_dates != [pd.Timestamp(d) for d in ["2014-03-01", "2015-11-01", "2021-10-01"]], (
        f"{where}, the end column holds the first date of the next regime. Record the regime's own "
        "last date: segment.index[-1].")
    assert starts == WANT_STARTS and last_dates == WANT_ENDS, (
        f"{where}, the regimes run from {[str(d.date()) for d in starts]} to "
        f"{[str(d.date()) for d in last_dates]}; they should run from "
        f"{[str(d.date()) for d in WANT_STARTS]} to {[str(d.date()) for d in WANT_ENDS]}. Record each "
        "regime's first and last dates: segment.index[0] and segment.index[-1].")

    means, stds = list(table["mean"]), list(table["std"])

    def rounded(got, want):
        """True if got is want rounded to between 1 and 6 decimals, and not want itself."""
        return not all_near(got, want) and any(
            len(got) == len(want) and all(abs(a - round(b, k)) < 1e-12 for a, b in zip(got, want))
            for k in range(1, 7))

    for label, got, want in [("means", means, [0.7170785690, 2.9529374794, -3.7383363564]),
                             ("standard deviations", stds, [1.0154054972, 0.3691426284, 2.3582819393])]:
        assert not rounded(got, want), (
            f"{where}, the {label} are rounded ({got}). Do not round inside the function: the check "
            "compares about six significant digits. Round only when you print.")
    assert not all_near(means, [-0.9686466265] * 3), (
        f"{where}, every row has the mean of the whole series. Take each regime's own mean.")
    assert all_near(means, [0.7170785690, 2.9529374794, -3.7383363564]), (
        f"{where}, the means are {[f'{m:.8f}' for m in means]}; they should be [0.71707857, "
        "2.95293748, -3.73833636], each regime's own mean, from pandas' .mean(). Do not round.")
    assert not all_near(stds, [1.0052001578, 0.3597957283, 2.3385470154]), (
        f"{where}, the standard deviations divide by n, as np.std does. Use pandas' .std(), which "
        "divides by n - 1, as Part 5's cell did.")
    assert all_near(stds, [1.0154054972, 0.3691426284, 2.3582819393]), (
        f"{where}, the standard deviations are {[f'{s:.8f}' for s in stds]}; they should be "
        "[1.01540550, 0.36914263, 2.35828194], each regime's own, from pandas' .std(). Do not round.")

    adf_ps, kpss_ps = list(table["adf_p"]), list(table["kpss_p"])
    WANT_ADF, WANT_KPSS = [7.3287422657e-09, 0.4167697216, 0.1807633171], [0.0234830229, 0.0858375100, 0.1]
    assert not (all_near(adf_ps, WANT_KPSS) and all_near(kpss_ps, WANT_ADF)), (
        f"{where}, the ADF and KPSS p-values are swapped. Take adf_p and kpss_p from the keys of "
        "the same names.")
    assert not all_near(adf_ps, [0.2683584878] * 3), (
        f"{where}, every row has the ADF p-value of the whole series. Run test_stationarity on each "
        "regime, not on series.")
    assert not (near(adf_ps[0], 2.4042573937e-09) and near(adf_ps[2], 0.4668046446)), (
        f"{where}, the p-values come from regression=\"ct\". A regime of a growth rate has no trend: "
        "call test_stationarity with its default regression, as Part 5's cell did.")
    assert all_near(adf_ps, WANT_ADF), (
        f"{where}, the ADF p-values are {[round(p, 4) for p in adf_ps]}; they should be [0.0000, "
        "0.4168, 0.1808]. Run test_stationarity on each regime with its default regression and take "
        "its \"adf_p\".")
    assert all_near(kpss_ps, WANT_KPSS), (
        f"{where}, the KPSS p-values are {[round(p, 4) for p in kpss_ps]}; they should be [0.0235, "
        "0.0858, 0.1000]. Take test_stationarity's \"kpss_p\".")
    decisions = list(table["decision"])
    assert decisions == ["contradictory", "inconclusive", "inconclusive"], (
        f"{where}, the decisions are {decisions}; with those p-values the 2x2 table gives "
        "['contradictory', 'inconclusive', 'inconclusive']. Take test_stationarity's \"decision\".")

    # 2. the same series with min_obs=30
    where = "On the made-up series with min_obs=30"
    sizes30 = [int(x) for x in run(where, made_up, [50, 70, 82, 142], 30)["n"]]
    assert sizes30 != [50, 20, 60], (
        f"{where}, regime_tests still keeps the 20-point regime: it uses 20, not its min_obs argument.")
    assert sizes30 == [50, 60], (
        f"{where}, regime_tests gives regimes of {sizes30} points; they should be [50, 60]. Does every "
        "step use min_obs?")

    # 3. GDP growth with Part 5's ends
    where = "On GDP growth with Part 5's ends"
    gdp_table = run(where, gdp_growth, ends)
    assert [int(x) for x in gdp_table["n"]] == [95, 65, 105], (
        f"{where}, regime_tests gives regimes of {list(gdp_table['n'])} quarters; they should be "
        "[95, 65, 105].")
    assert (all_near(list(gdp_table["mean"]), [0.8585338518, 0.8756483216, 0.5468040106])
            and all_near(list(gdp_table["std"]), [1.0758881510, 0.5060717614, 1.2379560673])
            and all_near(list(gdp_table["kpss_p"]), [0.1, 0.1, 0.1])
            and all_near(list(gdp_table["adf_p"]), [5.5563483749e-05, 6.5663285486e-04, 6.4022587215e-23])
            and list(gdp_table["decision"]) == ["stationary"] * 3), (
        f"{where}, regime_tests passed on the made-up series but gives other numbers here. Does it use "
        "anything other than its arguments?")

    print(f"Passed: regime_tests matches. Made-up series: regimes of {sizes} points with min_obs=20, "
          f"{sizes30} with min_obs=30.")
    print("GDP growth, Part 5's regimes:")
    print(gdp_table.round(4).to_string())

**Expected output:**
```
Passed: regime_tests matches. Made-up series: regimes of [50, 20, 60] points with min_obs=20, [50, 60] with min_obs=30.
GDP growth, Part 5's regimes:
       start        end    n    mean     std   adf_p  kpss_p    decision
0 1960-04-01 1983-10-01   95  0.8585  1.0759  0.0001     0.1  stationary
1 1984-01-01 2000-01-01   65  0.8756  0.5061  0.0007     0.1  stationary
2 2000-04-01 2026-04-01  105  0.5468  1.2380  0.0000     0.1  stationary
```

---

## Part 6: Production Module — `decompose.py` (10 min)

Here is the module, finished except for one function. It keeps what this lab found:
`run_stl` decomposes on the log scale unless told not to, and refuses a series it cannot take the
log of; `test_stationarity` passes one regression to both tests and reads them together in the 2×2
table; `detect_breaks` returns PELT's `ends`; and `regime_report` hands them to your
`regime_tests`, so that one call goes from a series to its table of regimes. `two_by_two` and
`test_stationarity` are Part 2's.

**Your task.** Paste your `regime_tests` from Write it yourself, unchanged, at the end of the first
cell, where it says so. Then run the two cells, compare the output with the expected output, and
answer the question under it. The first cell saves the module as a file; the second imports it and
runs this lab's main results through it.

**Reading the cells.** `%%writefile src/decompose.py` saves the cell as that file instead of
running it. The notes after a colon or `->` (`period: int`, `-> dict`) only say what type each
argument and result is, and `raise ValueError(...)` stops with that message. In the second cell,
`sys.path.insert(0, "src")` lets Python find modules in the `src` folder, `import decompose` loads
the file as a module, `importlib.reload` loads it again so that a re-run of the `%%writefile` cell
takes effect, and `hasattr(decompose, "regime_tests")` asks whether the module defines that name.

In [ ]:
%%writefile src/decompose.py
"""Decompose, test and split an economic time series (ECON 5200 Lab 20).

A seasonal pattern that grows with the level of a series is decomposed on the log scale. ADF and
KPSS are run with the same regression and read together. A series with structural breaks is tested
regime by regime, because one test over the whole series averages across the breaks.
"""
import numpy as np
import pandas as pd
import ruptures as rpt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss


def run_stl(series: pd.Series, period: int = 12, log_transform: bool = True, robust: bool = True):
    """STL of the series, on the log scale unless log_transform is False."""
    if log_transform:
        if (series <= 0).any():
            raise ValueError("log_transform=True needs a series that is positive everywhere")
        series = np.log(series)
    return STL(series, period=period, robust=robust).fit()


def two_by_two(adf_p: float, kpss_p: float, alpha: float = 0.05) -> str:
    """The 2x2 table's decision for an ADF and a KPSS p-value at level alpha."""
    if adf_p < alpha and kpss_p >= alpha:
        return "stationary"
    if adf_p >= alpha and kpss_p < alpha:
        return "non-stationary"
    if adf_p < alpha and kpss_p < alpha:
        return "contradictory"
    return "inconclusive"


def test_stationarity(series: pd.Series, regression: str = "c", alpha: float = 0.05) -> dict:
    """ADF and KPSS on one series with the same regression, and the 2x2 table's decision."""
    adf = adfuller(series, regression=regression)
    kpss_result = kpss(series, regression=regression, nlags="auto")
    return {"adf_stat": adf[0], "adf_p": adf[1], "kpss_stat": kpss_result[0],
            "kpss_p": kpss_result[1], "decision": two_by_two(adf[1], kpss_result[1], alpha)}


def detect_breaks(series: pd.Series, pen: float = 2, model: str = "rbf") -> list:
    """PELT's end position for each regime; the last is len(series)."""
    return rpt.Pelt(model=model).fit(series.values).predict(pen=pen)


def regime_report(series: pd.Series, pen: float = 2, model: str = "rbf", min_obs: int = 20) -> pd.DataFrame:
    """The regimes PELT finds in the series, each tested by regime_tests."""
    return regime_tests(series, detect_breaks(series, pen, model), min_obs)


# Paste your regime_tests function (the required exercise after Part 5) below this line, unchanged.

In [ ]:
# GIVEN — run as-is, once your regime_tests is pasted into the module
# Import the module and run this lab's main results through it
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import decompose
importlib.reload(decompose)        # reload, so a re-run of the %%writefile cell takes effect

if not hasattr(decompose, "regime_tests"):
    print("Paste your regime_tests into the module cell above first, then run that cell and this "
          "one again.")
else:
    retail_swings = seasonal_swing(decompose.run_stl(retail))
    print(f"run_stl(retail): ratio of the 2025 swing to the 2000 swing {retail_swings[2025] / retail_swings[2000]:.2f}")
    for name, series, regression in [("log GDP", log_gdp, "ct"), ("trend_demo", trend_demo, "ct"),
                                     ("GDP growth", gdp_growth, "c")]:
        print(f"test_stationarity({name}, {regression!r}): "
              f"{decompose.test_stationarity(series, regression)['decision']}")
    print(f"detect_breaks(GDP growth): {decompose.detect_breaks(gdp_growth)}")
    print(decompose.regime_report(gdp_growth).round(4).to_string())

**Expected output:**
```
run_stl(retail): ratio of the 2025 swing to the 2000 swing 0.90
test_stationarity(log GDP, 'ct'): non-stationary
test_stationarity(trend_demo, 'ct'): stationary
test_stationarity(GDP growth, 'c'): stationary
detect_breaks(GDP growth): [95, 160, 265]
       start        end    n    mean     std   adf_p  kpss_p    decision
0 1960-04-01 1983-10-01   95  0.8585  1.0759  0.0001     0.1  stationary
1 1984-01-01 2000-01-01   65  0.8756  0.5061  0.0007     0.1  stationary
2 2000-04-01 2026-04-01  105  0.5468  1.2380  0.0000     0.1  stationary
```
`{regression!r}` prints the string with its quotes.

### YOUR DIAGNOSIS

1. Every regime in your table is stationary, and so was GDP growth as a whole in Part 2. What did PELT find that the stationarity tests did not? Use the table's means and standard deviations, and say what it means for a model of GDP growth estimated on all of 1960–2026 and used to forecast 2027.

*Your answers here:*

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR ...]` with your own words or the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Fixed an STL decomposition of U.S. retail sales (FRED RSXFSN,
  2000 to June 2026): [YOUR DIAGNOSIS, IN ONE LINE]. The ratio of
  the 2025 seasonal swing to the 2000 one went from [YOUR VALUE]
  to [YOUR VALUE]
* Fixed a pair of ADF and KPSS tests on log real GDP (GDPC1, from
  1960): [YOUR DIAGNOSIS, IN ONE LINE]. Fixed, they read GDP as
  [YOUR RESULT]
* Separated two cycles in simulated hourly demand with MSTL, and put
  a moving block bootstrap band on the trend of log GDP
* Found breaks in GDP growth with PELT after [YOUR DATES], and wrote
  a function that tests each regime: [YOUR RESULT]
* Shipped the module decompose.py

**Please write a README.md entry including:**
1. Project Title: Time Series Diagnostics and Decomposition
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab20-time-series`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab20-time-series`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab20-time-series`, branch `main`,
   commit message `Lab 20 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/decompose.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.